In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
'''
@file          : test_timing_propagation.py
@author        : AI Assistant
@brief         : Unit tests for timing_propagation.py
@version       : 1.2 (Added converging path test case)
@date          : 2025-04-21
'''
import unittest
import torch
import timing_propagation # Import the module to be tested
import torch
import torchvision.models as models
from torch.utils.tensorboard import SummaryWriter

# --- Redefine/Import Mock Objects and Functions ---
MockTimingArc = timing_propagation.MockTimingArc
MockLibCell = timing_propagation.MockLibCell

# --- UPDATED Mock Library with AND2 Gate ---
mock_lib = {
    10: MockLibCell("INV"),
    11: MockLibCell("BUF"),
    12: MockLibCell("AND2") # New AND2 gate
}
# INV arc (Input -> Output)
mock_lib[10].add_arc(100, MockTimingArc(base_delay=0.5, tran_coeff=0.1, cap_coeff=0.5))
# BUF arc (Input -> Output)
mock_lib[11].add_arc(100, MockTimingArc(base_delay=0.4, tran_coeff=0.1, cap_coeff=0.4))
# AND2 arcs
# Arc: Input A -> Output Z (use descriptive index 120)
mock_lib[12].add_arc(120, MockTimingArc(base_delay=0.6, tran_coeff=0.12, cap_coeff=0.6))
# Arc: Input B -> Output Z (use descriptive index 121)
mock_lib[12].add_arc(121, MockTimingArc(base_delay=0.65, tran_coeff=0.13, cap_coeff=0.65)) # Slightly different delay

# Override the imported mock_getLibCellInfo to use this updated library
def mock_getLibCellInfo_override(cell_idx, corner):
    # Ignores corner for this simple mock
    return mock_lib.get(cell_idx, None)

timing_propagation.mock_getLibCellInfo = mock_getLibCellInfo_override # Use the updated lookup

# --- Refined Mock Calculation Functions ---
# (Keep the same refined mock functions using the now updated mock_lib)
def mock_lookup_and_calculate(arc_lib_idx_tensor, cell_lib_idx_tensor, tran_tensor, cap_tensor):
    results = []
    if arc_lib_idx_tensor.ndim == 0: arc_lib_idx_tensor = arc_lib_idx_tensor.unsqueeze(0)
    if cell_lib_idx_tensor.ndim == 0: cell_lib_idx_tensor = cell_lib_idx_tensor.unsqueeze(0)
    if tran_tensor.ndim == 0: tran_tensor = tran_tensor.unsqueeze(0)
    if cap_tensor.ndim == 0: cap_tensor = cap_tensor.unsqueeze(0)

    arc_indices = arc_lib_idx_tensor.cpu().tolist()
    cell_indices = cell_lib_idx_tensor.cpu().tolist()
    trans = tran_tensor.cpu().tolist()
    caps = cap_tensor.cpu().tolist()

    for arc_idx, cell_idx, tran, cap in zip(arc_indices, cell_indices, trans, caps):
        # Use the OVERRIDDEN lookup function here
        lib_cell = timing_propagation.mock_getLibCellInfo(int(cell_idx), None)
        arc = lib_cell.timingArcs.get(int(arc_idx)) if lib_cell else None
        if arc:
            results.append(arc.calculate(tran, cap))
        else:
            print(f"Warning: Mock arc not found for cell_idx={cell_idx}, arc_idx={arc_idx}. Returning 0.0")
            results.append(0.0)

    return torch.tensor(results, device=tran_tensor.device, dtype=torch.float32)

# Patch the original functions (Do this *outside* the class)
timing_propagation.r_delay_entry = lambda arc_lib_idx, cell_lib_idx, tran, cap: mock_lookup_and_calculate(arc_lib_idx, cell_lib_idx, tran, cap)
timing_propagation.f_delay_entry = lambda arc_lib_idx, cell_lib_idx, tran, cap: mock_lookup_and_calculate(arc_lib_idx, cell_lib_idx, tran, cap)
timing_propagation.r_tran_entry = lambda arc_lib_idx, cell_lib_idx, tran, cap: mock_lookup_and_calculate(arc_lib_idx, cell_lib_idx, tran, cap)
timing_propagation.f_tran_entry = lambda arc_lib_idx, cell_lib_idx, tran, cap: mock_lookup_and_calculate(arc_lib_idx, cell_lib_idx, tran, cap)


# --- Test Class ---

class TestTimingPropagation(unittest.TestCase):

    def setUp(self):
        """Set up test data for a circuit with converging paths."""
        # Circuit:
        # PI_A(0) -> Net0 -> INV(C0, P2_in, P3_out) -> Net2 -> AND2(C2, P6_inA) -----\
        #                                                                            -> AND2(P8_outZ) -> Net4 -> PO(9)
        # PI_B(1) -> Net1 -> BUF(C1, P4_in, P5_out) -> Net3 -> AND2(C2, P7_inB) -----/

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"\nSetting up for CONVERGING test on device: {self.device}")

        self.num_pins = 10 # 0,1:PI, 2:InvIn, 3:InvOut, 4:BufIn, 5:BufOut, 6:AndInA, 7:AndInB, 8:AndOut, 9:PO
        self.start_points = torch.tensor([0, 1], dtype=torch.long, device=self.device)
        self.end_points = torch.tensor([9], dtype=torch.long, device=self.device) # Single PO

        # --- Input conditions ---
        self.inrdelays = torch.tensor([0.1, 0.12], dtype=torch.float32, device=self.device) # AAT at PI
        self.infdelays = torch.tensor([0.15, 0.16], dtype=torch.float32, device=self.device) # AAT at PI
        self.inrtrans = torch.tensor([0.05, 0.06], dtype=torch.float32, device=self.device) # Tran at PI
        self.inftrans = torch.tensor([0.07, 0.08], dtype=torch.float32, device=self.device) # Tran at PI
        self.outcaps = torch.tensor([0.5], dtype=torch.float32, device=self.device) # Load cap at single PO (Pin 9)

        # --- Circuit Structure ---
        # Pin index:    0  1  2  3  4  5  6  7  8  9
        # Net index:    0  1  0  2  1  3  2  3  4  4
        self.pin_net = torch.tensor([0, 1, 0, 2, 1, 3, 2, 3, 4, 4], dtype=torch.long, device=self.device)

        # Cells: 0 (INV), 1 (BUF), 2 (AND2)
        self.cells_by_level = [
            torch.tensor([0, 1], dtype=torch.long, device=self.device), # Level 0: INV, BUF
            torch.tensor([2], dtype=torch.long, device=self.device)     # Level 1: AND2
        ]
        self.cells_by_reverse_level = [
            torch.tensor([2], dtype=torch.long, device=self.device),     # Reverse Level 0: AND2
            torch.tensor([0, 1], dtype=torch.long, device=self.device)  # Reverse Level 1: INV, BUF
        ]

        self.cell_flat_outpins = torch.tensor([3, 5, 8], dtype=torch.long, device=self.device) # Output pins for C0, C1, C2
        self.cell_flat_outpins_start = torch.tensor([0, 1, 2, 3], dtype=torch.long, device=self.device) # Starts for C0, C1, C2, End

        # Net Arcs: [driver_pin_idx, load_pin_idx]
        # Net 0: 0 -> 2
        # Net 1: 1 -> 4
        # Net 2: 3 -> 6
        # Net 3: 5 -> 7
        # Net 4: 8 -> 9
        self.net_flat_arcs = torch.tensor([
            [0, 2], [1, 4], [3, 6], [5, 7], [8, 9]
        ], dtype=torch.long, device=self.device)
        self.net_flat_arcs_start = torch.tensor([0, 1, 2, 3, 4, 5], dtype=torch.long, device=self.device) # Starts for Net 0,1,2,3,4, End

        # Cell Arcs: [in_pin_idx, out_pin_idx, arc_lib_idx, cell_lib_idx]
        # Cell 0 (INV): Pin 2 -> Pin 3, Arc 100, Type 10
        # Cell 1 (BUF): Pin 4 -> Pin 5, Arc 100, Type 11
        # Cell 2 (AND): Pin 6 -> Pin 8, Arc 120, Type 12
        # Cell 2 (AND): Pin 7 -> Pin 8, Arc 121, Type 12  <- Convergence on Pin 8
        self.cell_flat_arcs = torch.tensor([
            [2, 3, 100, 10],
            [4, 5, 100, 11],
            [6, 8, 120, 12],
            [7, 8, 121, 12]
        ], dtype=torch.long, device=self.device)
         # Starts: Cell 0 (1 arc), Cell 1 (1 arc), Cell 2 (2 arcs), End
        self.cell_flat_arcs_start = torch.tensor([0, 1, 2, 4], dtype=torch.long, device=self.device)

        # --- Input Tensors for forward() call (Base values) ---
        self.pin_net_delay_base = torch.zeros(self.num_pins, dtype=torch.float32, device=self.device)
        # Add wire delays to load pins of nets
        self.pin_net_delay_base[2] = 0.02 # Net 0 load (Inv Input)
        self.pin_net_delay_base[4] = 0.03 # Net 1 load (Buf Input)
        self.pin_net_delay_base[6] = 0.04 # Net 2 load (And Input A)
        self.pin_net_delay_base[7] = 0.05 # Net 3 load (And Input B)
        self.pin_net_delay_base[9] = 0.06 # Net 4 load (PO)

        self.pin_net_impulse_base = torch.zeros(self.num_pins, dtype=torch.float32, device=self.device)
        self.pin_net_impulse_base[2] = 0.01
        self.pin_net_impulse_base[4] = 0.015
        self.pin_net_impulse_base[6] = 0.02
        self.pin_net_impulse_base[7] = 0.025
        self.pin_net_impulse_base[9] = 0.03

        # Pin capacitance: Input cap for cell inputs, effective cap for cell outputs driving nets
        self.pin_net_cap_base = torch.zeros(self.num_pins, dtype=torch.float32, device=self.device)
        self.pin_net_cap_base[2] = 0.1  # Inv Input cap
        self.pin_net_cap_base[4] = 0.12 # Buf Input cap
        self.pin_net_cap_base[6] = 0.15 # And Input A cap
        self.pin_net_cap_base[7] = 0.16 # And Input B cap
        # Effective cap seen by output pins driving nets
        self.pin_net_cap_base[3] = 0.15 # INV Out (Pin 3) driving Net 2 -> Pin 6 Cap
        self.pin_net_cap_base[5] = 0.16 # BUF Out (Pin 5) driving Net 3 -> Pin 7 Cap
        self.pin_net_cap_base[8] = 0.5  # AND Out (Pin 8) driving Net 4 -> Pin 9 Cap (PO cap)
        # PO cap (Pin 9) will be set by self.outcaps in forward pass

        # --- Required times for RAT calculation (arbitrary for test) ---
        self.output_required_rtime = torch.tensor(10.0, dtype=torch.float32, device=self.device) # Increased required time
        self.output_required_ftime = torch.tensor(10.0, dtype=torch.float32, device=self.device) # Increased required time


        # --- Model Instance ---
        self.model = timing_propagation.TimingPropagation(
            inrdelays=self.inrdelays,
            infdelays=self.infdelays,
            inrtrans=self.inrtrans,
            inftrans=self.inftrans,
            outcaps=self.outcaps,
            pin_net=self.pin_net,
            cells_by_level=self.cells_by_level,
            cell_flat_outpins=self.cell_flat_outpins,
            cell_flat_outpins_start=self.cell_flat_outpins_start,
            start_points=self.start_points,
            end_points=self.end_points,
            net_flat_arcs_start=self.net_flat_arcs_start,
            net_flat_arcs=self.net_flat_arcs,
            cell_flat_arcs_start=self.cell_flat_arcs_start,
            cell_flat_arcs=self.cell_flat_arcs,
            cells_by_reverse_level=self.cells_by_reverse_level
        )

        # --- Patch Forward Method (Apply necessary fixes for testing) ---
        # self.model.forward = self._patched_forward # Replace with patched version

    def test_forward_pass(self):
        """Tests the forward pass calculation (WNS, TNS)."""
        print("\n--- Running test_forward_pass (Converging Circuit) ---")
        wns, tns = self.model.forward(
            self.pin_net_delay_base,
            self.pin_net_impulse_base,
            self.pin_net_cap_base
        )

        print(f"Forward Pass WNS: {wns.item()}")
        print(f"Forward Pass TNS: {tns.item()}")

        # --- Assertions ---
        self.assertIsInstance(wns, torch.Tensor)
        self.assertIsInstance(tns, torch.Tensor)
        self.assertEqual(wns.ndim, 0)
        self.assertEqual(tns.ndim, 0)
        self.assertEqual(wns.dtype, torch.float32)
        self.assertEqual(tns.dtype, torch.float32)
        self.assertLessEqual(wns.item(), 0.0) # Assume potential violations
        self.assertLessEqual(tns.item(), 0.0)


    def test_backward_pass(self):
        """Tests the backward pass (gradient computation)."""
        print("\n--- Running test_backward_pass (Converging Circuit) ---")

        pin_net_delay = self.pin_net_delay_base.clone().detach().requires_grad_(True)
        pin_net_impulse = self.pin_net_impulse_base.clone().detach().requires_grad_(True)
        pin_net_cap = self.pin_net_cap_base.clone().detach().requires_grad_(True)
        writer = SummaryWriter('logs/timing_propagation')
        wns, tns = self.model.forward(pin_net_delay, pin_net_impulse, pin_net_cap)
        writer.add_graph(self.model, (pin_net_delay, pin_net_impulse, pin_net_cap))
        writer.close()
        print(f"Backward Test - Forward WNS: {wns.item()}")
        print(f"Backward Test - Forward TNS: {tns.item()}")

        loss = tns

        pin_net_delay.grad = None
        pin_net_impulse.grad = None
        pin_net_cap.grad = None

        try:
            loss.backward()
            print("loss.backward() executed.")
        except Exception as e:
            self.fail(f"loss.backward() raised an exception: {e}")

        # --- Assertions for Gradients ---
        self.assertIsNotNone(pin_net_delay.grad, "Gradient for pin_net_delay should exist")
        self.assertIsNotNone(pin_net_impulse.grad, "Gradient for pin_net_impulse should exist")
        self.assertIsNotNone(pin_net_cap.grad, "Gradient for pin_net_cap should exist")

        self.assertEqual(pin_net_delay.grad.shape, pin_net_delay.shape)
        self.assertEqual(pin_net_impulse.grad.shape, pin_net_impulse.shape)
        self.assertEqual(pin_net_cap.grad.shape, pin_net_cap.shape)

        atol = 1e-7
        self.assertTrue(torch.any(torch.abs(pin_net_delay.grad) > atol), "Gradients for pin_net_delay are all zero/close to zero.")
        # Note: Impulse gradient might be zero if tran calc doesn't depend strongly on it or if path isn't critical
        self.assertTrue(torch.any(torch.abs(pin_net_impulse.grad) > atol), "Gradients for pin_net_impulse are all zero/close to zero.")
        self.assertTrue(torch.any(torch.abs(pin_net_cap.grad) > atol), "Gradients for pin_net_cap are all zero/close to zero.")

        print(f"Gradient for pin_net_delay (sum): {torch.sum(pin_net_delay.grad).item()}")
        print(f"Gradient for pin_net_impulse (sum): {torch.sum(pin_net_impulse.grad).item()}")
        print(f"Gradient for pin_net_cap (sum): {torch.sum(pin_net_cap.grad).item()}")

        # Print gradient for a specific converging pin's input delay/cap if needed
        # print(f"Gradient pin_net_delay[6]: {pin_net_delay.grad[6].item()}") # Delay to AND InA
        # print(f"Gradient pin_net_delay[7]: {pin_net_delay.grad[7].item()}") # Delay to AND InB